In [1]:
#!pip install update pyvisa

INITIALIZE

In [1]:
import pyvisa
import numpy as np
import time
import matplotlib.pyplot as plt
import os
import csv
%matplotlib qt

In [2]:
def update_plot(fig, ax, line, lstVoltage, lstAbsCurrent):
    line.set_data(lstVoltage, lstAbsCurrent)
    ax.relim()
    ax.autoscale_view()
    fig.canvas.draw()
    fig.canvas.flush_events()

In [3]:

rm = pyvisa.ResourceManager()

In [6]:
def fncMeasureOne(arrVMeasurement, \
                  device= 'pico',\
                  Sample = 'Test', sweepNum = 1,\
                  folderPath='230606\\Dev02\\Test',\
                  askToStart=False):
    if (device == 'pico'):
        fncMeasureOnePico(arrVMeasurement, Sample, sweepNum, folderPath, askToStart)
    elif (device == 'keith'):
        fncMeasureOneKL(arrVMeasurement, Sample, sweepNum, folderPath, askToStart)
    elif (device == 'keith1'):
        fncMeasureOneKL1(arrVMeasurement, Sample, sweepNum, folderPath, askToStart)
    elif (device == 'M2657A'):
        fncMeasureOneM2657A(arrVMeasurement, Sample, sweepNum, folderPath, askToStart)
    elif (device == 'M2461SWETSP'): #TSP means using TSP commands, SWE means sweep the whole thing from start to stop in one run
        fncMeasureOneM2461SWETSP(arrVMeasurement, Sample, sweepNum, folderPath, askToStart)
    elif (device == 'M2461POITSP'): #POI means set each point and then measure and move to another.
        fncMeasureOneM2461POITSP(arrVMeasurement, Sample, sweepNum, folderPath, askToStart)
    else:
        print("Incorrect device name")

In [7]:
def fncMeasureMultiple(settings =['pico', [[0, 0.5, 0.1]]], \
                  Sample = 'Test',\
                  startNum = 1, \
                  folderPath='230606\\Dev02\\Test',\
                  askToStart=False): 
    sweepNum = 0
    dev, arrVMeasurements = settings
    for j in range (len(arrVMeasurements)):
        sweepNum = j + startNum
        fncMeasureOne(arrVMeasurements[j], dev , Sample, sweepNum, folderPath)
        time.sleep(5)

### Model 2657A

In [9]:
#rm.close()
#rm = pyvisa.ResourceManager()
m2657A = rm.open_resource('TCPIP0::132.181.53.204::inst0::INSTR')
m2657A.timeout = 10000
m2657A.write('*RST')
# print(rm.last_status)


6

In [10]:
def setM2657AIVMeasurement (iRange = 0.02, iLimit = 0.02, nplc = 1, delay = 0.05):
    # Reset and initialize instrutment
    m2657A.write(f"reset()")
    m2657A.write(f"status.reset()")
    m2657A.write(f"errorqueue.clear()")
    
    # Configure source function as 2W DCVOLTS
    m2657A.write(f"smua.source.func = smua.OUTPUT_DCVOLTS")
    m2657A.write(f"smua.sense = smua.SENSE_LOCAL")
    m2657A.write(f"display.screen = display.SMUA")
    m2657A.write(f"display.smua.measure.func = display.MEASURE_DCAMPS")
    m2657A.write(f"display.smua.digits = display.DIGITS_5_5")

    #Set up current compliance
    m2657A.write(f"smua.source.limiti = %f" %iLimit)

    #Set up current range 
    #Set up current range
    if iRange == "auto":
        m2657A.write(f"smua.measure.autorangei = smua.AUTORANGE_ON")
    else:
        m2657A.write(f"smua.measure.autorangei = smua.AUTORANGE_OFF")
        m2657A.write(f"smua.measure.rangei = %f" %iRange)

    

    #Set the measurement integration time
    m2657A.write(f"smua.measure.nplc = %f" %nplc)
    m2657A.write(f"smua.measure.delay = %f" %delay)

    

    


    # m2657A.write(f"")
    # m2657A.write(f"")
    # m2657A.write(f"")
    # m2657A.write(f"")
    # m2657A.write(f"")
    # m2657A.write(f"")
    # m2657A.write(f"")
    # m2657A.write(f"")
	
    m2657A.write(f"beeper.beep(0.5,100)")
    return True
def measureM2657AOnePoint(setV, outputON = False, resetV = True):
    #setM2657AIVMeasurement()
    m2657A.write(f"smua.source.levelv = %f" %setV)
    
    #Configure the reading buffers
    m2657A.write(f"smua.nvbuffer1.clear()")
    m2657A.write(f"smua.nvbuffer1.appendmode = 1")
    m2657A.write(f"smua.nvbuffer1.collecttimestamps = 1")
    m2657A.write(f"smua.nvbuffer1.collectsourcevalues = 0")
    m2657A.write(f"smua.nvbuffer1.fillmode = smua.FILL_ONCE")
    m2657A.write(f"smua.nvbuffer2.clear()")
    m2657A.write(f"smua.nvbuffer2.appendmode = 1")
    m2657A.write(f"smua.nvbuffer2.collecttimestamps = 1")
    m2657A.write(f"smua.nvbuffer2.collectsourcevalues = 0")
    m2657A.write(f"smua.nvbuffer2.fillmode = smua.FILL_ONCE")
    
    
    if (not outputON):
        m2657A.write(f"smua.source.output = 1")
    m2657A.write(f"smua.measure.i(smua.nvbuffer1)")
    m2657A.write(f"smua.measure.v(smua.nvbuffer2)")
    #time.sleep(2)

    #Turn off the output
    if resetV:
        m2657A.write(f"smua.source.levelv = 0")
        m2657A.write(f"smua.source.output = 0")
    
    m2657A.write(f"printbuffer(1, 1, smua.nvbuffer1.readings, smua.nvbuffer2.readings)")
    rawData = m2657A.read()
    # print("%g\t%g\t%g", smua.nvbuffer1.timestamps[1], smua.nvbuffer1.readings[1], smua.nvbuffer2.readings[1])
    arrRawData = rawData.split(",")
    #print ("rawData: ", rawData)

    return [float(arrRawData[i]) for i in range(len(arrRawData))]
setM2657AIVMeasurement(0.02, 0.02, 1, 0.05)
# rur = measureM2657AOnePoint(1,False,True)
# print (results)
current, voltage = measureM2657AOnePoint(1,False,True)
print ("Current: ", current, "Voltage: ", voltage)

Current:  -1.24495e-07 Voltage:  0.998603


In [11]:
def fncMeasureOneM2657A(arrVMeasurement, \
                  Sample = 'Test', sweepNum = 1,\
                  folderPath='241108\\Dev02\\Test',\
                  askToStart=False,\
                  vRange = 200,\
                  iRange = "auto",\
                  iLimit = 2e-2,\
                  nplc = 1, \
                  delay = 0.05,\
                  breakDownVolt = 1500,\
                  breakDownCurr = 0.02):
    Vstart, Vstop, Vstep = arrVMeasurement
    fileName = "Sweep"+str(sweepNum)+Sample+'Star'+str(Vstart).replace("-","M")+'_Stop'+str(Vstop).replace("-","M")+'_Step'+str(Vstep).replace("-", "M")+'.csv'
    filePath = os.path.join(folderPath, fileName)
    print(filePath)
    stepNum = np.floor(abs((Vstart - Vstop)/Vstep)).astype(int) + 1
    # vRange = 500
    # vILIM = 2e-3
    # iRange = '2e-9'
    #iNPLC = 1
    #vILIM = 'SOUR:VOLT:ILIM' + vILIM
    #iRange = ':SENSE:CURR:RANG ' + iRange



    setM2657AIVMeasurement(iRange, iLimit, nplc, delay)
    absVstart = abs(Vstart)
    absVstop = abs(Vstop)
    # Set up source range
    if abs(Vstart) > abs(Vstop): 
        m2657A.write(f"smua.source.rangev = %f" %absVstart)
    else:
        m2657A.write(f"smua.source.rangev = %f" %absVstop)
    
    lstCurrent = []
    lstVoltage = []
    lstAbsCurrent = []
    
    fig, ax = plt.subplots()
    line, = ax.plot([], [], 'o-')
    ax.set_xlabel('Voltage (V)')
    ax.set_ylabel('Absolute Current (A)')
    ax.set_yscale('log')
    
    

    plt.show()
    time.sleep(1)
    m2657A.write(f"smua.source.output = 1")
    curExceedLimitMeasNum = 0 # number of times the current exceeds the current limit - error
    afterBreakDownMeasNum = 10 # number of additional measurements after the current reaches the current limit at the flat region
    curLimitError = 0.0003
    curLimitLowerBound = breakDownCurr - curLimitError
    # Measurement loop
    for i in range(stepNum):
        
        if np.absolute(Vstart + i*Vstep) > breakDownVolt:
            break
        #pico.write(':INIT:IMM')
        #strVol = '{0:.2f}'.format(i*0.2)
        #strVol = ':SOUR:VOLT {0:.2f}'.format(Vstart + i*Vstep)
        #pico.write(strVol)
        current, voltage = measureM2657AOnePoint(Vstart + i*Vstep,True,False)
        
        # results = pico.query(':MEASure:CURRent?')
        # #print(results)

        # current = results.split(',')[0][:-1]
        #voltage = l
        lstVoltage.append(voltage)
        lstCurrent.append(current)
        lstAbsCurrent.append(abs(current))
        update_plot(fig, ax, line,lstVoltage, lstAbsCurrent)
        if (abs((current)) >  breakDownCurr): #or (stop_flag == True)
            break
        if (abs(current) >  curLimitLowerBound):
            curExceedLimitMeasNum+=1
        if (curExceedLimitMeasNum >= afterBreakDownMeasNum):
            break
        time.sleep(0.1)
    #Turn off the output
    m2657A.write(f"smua.source.levelv = 0")
    m2657A.write(f"smua.source.output = 0")

    
    dataToWrite = [['vRange: ', vRange]]
    dataToWrite.append(['iLimit: ', iLimit])
    dataToWrite.append(['iRange: ', iRange[16:]])
    dataToWrite.append(['nplc: ', nplc, 'delay: ', delay])
    dataToWrite.append(['data Number:', stepNum])
    dataToWrite.append(['V_meas', 'I','abs_I'])
    # lstCalVoltage = [(Vstart + i*Vstep) for i in range(len(lstAbsCurrent))]

    for i in range(len(lstCurrent)):
        dataToWrite.append([lstVoltage[i], lstCurrent[i], lstAbsCurrent[i]])
    try:
        with open(filePath, 'w', newline = '') as file:
            writer = csv.writer(file)
            writer.writerows(dataToWrite)
            file.close()
    except IOError:
        print("Error: Could not open file for writing")
    plt.show()
    # plt.savefig(os.path.splitext(filePath)[0] + '.png')
    plt.savefig(os.path.splitext(filePath)[0] + '_log.png')
    ax.set_yscale('linear')
    plt.show()
    plt.savefig(os.path.splitext(filePath)[0] + '_lin.png')

In [12]:
m2657A.write(f"beeper.beep(2,100)")

20

In [8]:
folderPath = '../Devices/250502_Fab250424_AuAnode'
# Create a new directory
try:
    os.mkdir(folderPath)
except FileExistsError:
    print("Folder already exists")
    
folderPath = folderPath+ '\\Dev18'
# Create a new directory
try:
    os.mkdir(folderPath)
except FileExistsError:
    print("Folder already exists")

folderPath = folderPath+ '\\H01'
# Create a new directory
try:
    os.mkdir(folderPath)
except FileExistsError:
    print("Folder already exists")
sample = folderPath[-3:]

Folder already exists
Folder already exists
Folder already exists


In [14]:
#run M2657A
arrM2657AVMeasMulti =["M2657A", [[0,-1.5,-0.01],[0,1.5,0.01],[0,1500,1],[0,-1.5,-0.01],[0,1.5,0.01]]]
fncMeasureMultiple(arrM2657AVMeasMulti,sample, 7, folderPath)

../Devices/250502_Fab250424_AuAnode\Dev18\H01\Sweep7H01Star0_StopM1.5_StepM0.01.csv
../Devices/250502_Fab250424_AuAnode\Dev18\H01\Sweep8H01Star0_Stop1.5_Step0.01.csv
../Devices/250502_Fab250424_AuAnode\Dev18\H01\Sweep9H01Star0_Stop1500_Step1.csv
../Devices/250502_Fab250424_AuAnode\Dev18\H01\Sweep10H01Star0_StopM1.5_StepM0.01.csv
../Devices/250502_Fab250424_AuAnode\Dev18\H01\Sweep11H01Star0_Stop1.5_Step0.01.csv


In [13]:
#run M2657A
arrM2657AVMeasMulti =["M2657A", [[0,-1.5,-0.01],[0,1.5,0.01]]]
fncMeasureMultiple(arrM2657AVMeasMulti,sample, 5, folderPath)

../Devices/250502_Fab250424_AuAnode\Dev18\H01\Sweep5H01Star0_StopM1.5_StepM0.01.csv
../Devices/250502_Fab250424_AuAnode\Dev18\H01\Sweep6H01Star0_Stop1.5_Step0.01.csv


In [ ]:
m2657A.write(f"smua.source.levelv = 0")
m2657A.write(f"smua.source.output = 0")

In [15]:
rm.close()

In [ ]:
# Reset Model 2657A
rm = pyvisa.ResourceManager()
m2657A = rm.open_resource('TCPIP0::132.181.53.204::inst0::INSTR')
m2657A.timeout = 10000
m2657A.write('*RST')

In [21]:
#run M2657A
arrM2657AVMeasMulti =["M2657A", [[0,3000,100]]]
fncMeasureMultiple(arrM2657AVMeasMulti,sample, 3, folderPath)

241114_Fab231219\DTest\Air\Sweep3AirStar0_Stop3000_Step100.csv


In [ ]:
sweepFilePath = "C:\\Users\\gtd19\\Semsem\\bGa2O3\\m2461leyControl\\TSP_2657A\\2657A_DiodeRL\\DiodeRL.tsp"

m2657A.write("loadscript simpleSweep")
with open(sweepFilePath) as fp:
    for line in fp:
        m2657A.write(line)
m2657A.write("endscript")
m2657A.write("simpleSweep.run()")
fp.close()